<a href="https://colab.research.google.com/github/AkhilTeljeeru/Customer-churn-Prediction/blob/main/Customer_churn_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
df=pd.read_csv('/content/drive/MyDrive/Telco-Customer-Churn.csv')
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
y = df['Churn']
X = df.drop(['SeniorCitizen', 'Churn'], axis='columns')
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# Drop 'customerID' from x_train as it's an identifier
x_train_processed = x_train.drop('customerID', axis='columns')

# Encode categorical columns in x_train_processed
for column in x_train_processed.columns:
    if x_train_processed[column].dtype == 'object':
        le = LabelEncoder()
        x_train_processed[column] = le.fit_transform(x_train_processed[column])

# Encode y_train
le_y = LabelEncoder()
y_train_processed = le_y.fit_transform(y_train)

# Convert to PyTorch tensors
x_tensor = torch.tensor(x_train_processed.values, dtype=torch.float32)
y_tensor = torch.tensor(y_train_processed, dtype=torch.float32)

In [9]:
class ChurnDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
print(torch.isnan(x_tensor).sum())
print(torch.isnan(y_tensor).sum())

# Create an instance of the custom Dataset
dataset = ChurnDataset(x_tensor, y_tensor)

# Create the DataLoader
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Example of iterating through the loader
for x_batch, y_batch in loader:
  print("Batch X shape:", x_batch.shape)
  print("Batch Y shape:", y_batch.shape)
  break # Print only the first batch for brevity

tensor(0)
tensor(0)
Batch X shape: torch.Size([32, 18])
Batch Y shape: torch.Size([32])


In [10]:
class MyModel(nn.Module):
  def __init__(self):
    super(MyModel, self).__init__()
    # Input features should be 18, not 26, based on x_tensor's shape
    self.fc1 = nn.Linear(18, 64)
    self.fc2 = nn.Linear(64, 32)
    # For binary classification, output features should be 1
    self.fc3 = nn.Linear(32, 1)
    self.relu = nn.ReLU()
    # Removed self.sigmoid = nn.Sigmoid() as BCEWithLogitsLoss handles it

  def forward(self, x):
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    x = self.relu(x)
    x = self.fc3(x)
    # Removed x = self.sigmoid(x) as BCEWithLogitsLoss handles it
    return x

In [11]:
loss_fn=nn.BCEWithLogitsLoss()
model=MyModel()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [12]:
for x_batch, y_batch in loader:

    y_batch = y_batch.float().reshape(-1,1)

    outputs = model(x_batch)

    loss = loss_fn(
        outputs,
        y_batch
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

In [13]:
model.eval()
with torch.no_grad():
  for x,y in loader:
    predictions=model(x)
    loss=loss_fn(predictions, y.unsqueeze(1))
    print(f"Loss: {loss.item()}")

Loss: 0.9952086210250854
Loss: 0.9888863563537598
Loss: 0.9946721196174622
Loss: 0.7230634689331055
Loss: 0.7935312986373901
Loss: 1.012790322303772
Loss: 0.9305269718170166
Loss: 1.4336144924163818
Loss: 0.8915665745735168
Loss: 0.6796737909317017
Loss: 1.430943250656128
Loss: 0.8664920330047607
Loss: 0.772571861743927
Loss: 1.0339385271072388
Loss: 0.7316048741340637
Loss: 1.2474855184555054
Loss: 0.7271513342857361
Loss: 0.8298048377037048
Loss: 0.32218196988105774
Loss: 0.6709818243980408
Loss: 1.5015122890472412
Loss: 0.5731921792030334
Loss: 0.9511716365814209
Loss: 1.0046677589416504
Loss: 0.7019402980804443
Loss: 0.7086003422737122
Loss: 1.2177183628082275
Loss: 1.1909725666046143
Loss: 0.16860231757164001
Loss: 0.43437710404396057
Loss: 0.9749888181686401
Loss: 0.8608459234237671
Loss: 0.8315572738647461
Loss: 0.6399255990982056
Loss: 0.8518831133842468
Loss: 0.7476451396942139
Loss: 1.0814605951309204
Loss: 0.4080422520637512
Loss: 1.0325098037719727
Loss: 0.6101770401000977


In [ ]:
Sigmoid=nn.Sigmoid()
x=torch.tensor([1,2,3], dtype=torch.float32)
output=Sigmoid(x)
print(output)

tensor([0.7311, 0.8808, 0.9526])


In [14]:
model.eval()

predictions = []
targets = []

with torch.no_grad():
    for x_batch, y_batch in loader:

        outputs = model(x_batch)

        probs = torch.sigmoid(outputs)

        preds = (probs >= 0.5).float()

        predictions.append(preds.squeeze())
        targets.append(y_batch)

predictions = torch.cat(predictions)
targets = torch.cat(targets)

correct = (predictions == targets)

accuracy = torch.mean(correct.float())

print(accuracy)

tensor(0.7393)
